# 5 · Restricting recall to topics

Engram files each memory under a **topic**. Pass `topics` to restrict search to
specific topics — set a default at construction and/or override per invocation via
the runtime context (key `topics`, configurable with `topics_context_key`). Works
on `EngramMiddleware`, `create_memory_tools`, and `EngramStore`.

A common pattern is to keep a per-conversation summary in a `ConversationSummary`
topic and bind recall to **only** that topic, so the agent reads a clean summary
instead of scattered facts.

> **Note.** Topic names come from your project's group/pipeline configuration. The
> `ConversationSummary` topic used below must exist in your Engram project; inspect
> the topics on your own memories (first cell) to see what is available.

## Setup — pick **one** option

**Option A — editable install (recommended).** Installs this package *and* its
dependencies from source; edits to `langchain_engram/` are picked up after a
kernel restart.

**Option B — import from local source (no install of `langchain-engram`).** Adds
the repo root to `sys.path` so `import langchain_engram` resolves straight to the
source tree. You still need the runtime dependencies available — the easiest is to
launch Jupyter from the repo's environment: `uv run --with jupyter jupyter lab`.
> For the agent cells you also need `langchain-anthropic` (Option A installs it; for Option B run `%pip install langchain-anthropic`).

In [ ]:
# Option A — editable install from source.
%pip install -q -e ".." langchain-anthropic

In [ ]:
# Option B — use langchain-engram straight from the local source tree (no install).
# Run this INSTEAD of Option A.
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # notebooks/ -> repo root (holds langchain_engram/)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import langchain_engram

print('langchain_engram imported from:', langchain_engram.__file__)

In [ ]:
import getpass
import os

if not os.environ.get("ENGRAM_API_KEY"):
    os.environ["ENGRAM_API_KEY"] = getpass.getpass("ENGRAM_API_KEY: ")

# Optional: point at a non-default Engram endpoint.
# os.environ["ENGRAM_BASE_URL"] = "https://..."

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')

In [ ]:
import time

from langchain_engram._client import build_client

client = build_client()  # reads ENGRAM_API_KEY

# If your Engram group has SCOPED TOPICS, every WRITE must include the required
# scope properties, or the API rejects it with, e.g.:
#   APIError: group 'default': insufficient scope: missing required scope
#   properties [custom_scope_1 some_other] to write memories
# Fill in the scope properties your project requires (leave {} if none):
SCOPE_PROPERTIES = {}  # e.g. {'custom_scope_1': 'demo', 'some_other': 'demo'}


def _merge_props(properties):
    merged = {**SCOPE_PROPERTIES, **(properties or {})}
    return merged or None


def seed_memory(text, user_id, *, properties=None, **kwargs):
    """Add a memory (with required scope properties) and wait for it to commit."""
    run = client.memories.add(
        text, user_id=user_id, properties=_merge_props(properties), **kwargs
    )
    status = client.runs.wait(run.run_id, timeout=60.0)
    print(f'run {run.run_id}: {status.status} '
          f'(+{len(status.memories_created)} / ~{len(status.memories_updated)})')
    return status


def wait_until_searchable(query, user_id, needle, timeout=60.0, *, properties=None, **kwargs):
    """Poll search until `needle` shows up (handles eventual consistency)."""
    props = _merge_props(properties)
    deadline = time.time() + timeout
    while time.time() < deadline:
        results = client.memories.search(
            query=query, user_id=user_id, properties=props, **kwargs
        )
        if any(needle.lower() in m.content.lower() for m in results):
            return results
        time.sleep(2.0)
    return client.memories.search(
        query=query, user_id=user_id, properties=props, **kwargs
    )

### Inspect which topics Engram assigned to a user's memories

In [ ]:
import uuid

USER = f'topics-demo-{uuid.uuid4().hex[:8]}'
seed_memory('The user prefers window seats and decaf coffee.', user_id=USER)

hits = client.memories.search(query='preferences', user_id=USER)
topics_seen = sorted({m.topic for m in hits})
print('topics:', topics_seen)

### Bind middleware recall to one topic
Pin recall to the `ConversationSummary` topic. Set `RECALL_TOPIC` to a topic that
exists in your project (see the list above).

In [ ]:
from langchain.agents import create_agent

from langchain_engram import EngramMiddleware

RECALL_TOPIC = 'ConversationSummary'

agent = create_agent(
    'anthropic:claude-sonnet-4-6',
    middleware=[EngramMiddleware(user_id=USER, topics=[RECALL_TOPIC])],
)
out = agent.invoke(
    {'messages': [{'role': 'user', 'content': 'What do you know about me?'}]}
)
print(out['messages'][-1].content)

### Bind the search tools to the same topic
`create_memory_tools` takes the same `topics` argument, so the agent's
`search_memories` tool only ever reads `ConversationSummary` memories.

In [ ]:
from langchain_engram import create_memory_tools

search_tool, _ = create_memory_tools(user_id=USER, topics=[RECALL_TOPIC])
print('search tool bound to topic:', RECALL_TOPIC)
print(search_tool.invoke({'query': 'preferences'}))

### Override topics per invocation
Pass a `topics` list in the runtime context to temporarily change which topics
recall reads from — overriding the construction-time default.

In [ ]:
from typing import Optional, TypedDict


class TopicsCtx(TypedDict):
    user_id: str
    topics: Optional[list[str]]


topic_agent = create_agent(
    'anthropic:claude-sonnet-4-6',
    middleware=[EngramMiddleware(topics=[RECALL_TOPIC])],  # default
    context_schema=TopicsCtx,
)
out = topic_agent.invoke(
    {'messages': [{'role': 'user', 'content': 'What do you know about me?'}]},
    context={'user_id': USER, 'topics': topics_seen or [RECALL_TOPIC]},
)
print(out['messages'][-1].content)